# 01 · export v2 — Z: pickles → canonical parquet

**Kernel: `fttl-v2` (env-v2, Python 3.10, xgboost 1.4.2).** Reads v2's raw extract and the three
timestamped transformed splits off `Z:`, recomputes scores with the repo's own `model.pkl`
(v2 saved no train-time scores), converts the **production log** to the canonical log parquet
(the job `scoring/ingest.py` used to have — absorbed here 2026-08-09), and writes everything
into `src/data/real/`.

Blocking unknowns this notebook will surface on first run: the **timestamps** in the transformed
filenames, **v2's claim id column** (config placeholder — the inspect cell prints the columns),
and the **production log's location** (`config paths.log_source`).


In [ ]:
import sys
from pathlib import Path

import joblib
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import config                      # noqa: E402
import schema                      # noqa: E402

import xgboost                     # noqa: E402
assert xgboost.__version__ == config.xgboost_pin("v2"), (
    xgboost.__version__, "expected", config.xgboost_pin("v2"))

# ---- SOURCES: fill the real Z: paths (timestamps must be read off the Z: folder).
# ---- They live HERE, not in config — a declared config path would point the analysis
# ---- .venv at a version-bound pickle it cannot open. --------------------------------
RAW = r"Z:\...\Data\clean_dataset.pkl"
TRANSF = {
    "train": r"Z:\...\Data\train_transf_<ts>.pkl",
    "val":   r"Z:\...\Data\val_transf_<ts>.pkl",
    "test":  r"Z:\...\Data\test_transf_<ts>.pkl",
}


In [ ]:
raw = pd.read_pickle(RAW)
print(raw.shape)
print(list(raw.columns))
# → identify the claim id column here, then fill config.VERSIONS['v2']['columns']['claim_id']
#   (and re-run from the top). This notebook refuses to guess it.
ID = config.column("v2", "claim_id")     # raises while the placeholder is unfilled — on purpose
DATE, OBSERVED = config.column("v2", "date"), config.column("v2", "observed")


In [ ]:
# transformed splits -> one matrix with a split column (their split names kept: v2's "Test"=OOT)
parts = []
for split, p in TRANSF.items():
    d = pd.read_pickle(p)
    d["split"] = split
    parts.append(d)
proc = pd.concat(parts, ignore_index=True)
print(proc.shape, proc["split"].value_counts().to_dict())
assert ID in proc.columns, ID + " not in the transformed table — check which column carries it"

est = joblib.load(config.path("model", "v2", "real"))     # needs repo_dir declared in config
booster_cols = list(est.get_booster().feature_names)
scores = pd.DataFrame({
    "claim_id": proc[ID].values,
    "model_v2_score": est.predict_proba(proc[booster_cols])[:, 1],
})


In [ ]:
def write(df, kind):
    p = config.path(kind, "v2", "real")     # undeclared kinds fall back to src/data/real/…
    p.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(p, index=False)
    print(kind, "->", p, len(df), "rows")

write(raw.rename(columns={ID: "claim_id"}), "raw_dataset")
write(proc.rename(columns={ID: "claim_id"}), "processed_inputs")
write(raw[[ID, DATE, OBSERVED]].rename(
    columns={ID: "claim_id", DATE: "date", OBSERVED: "observed"}), "targets")
write(scores, "scores")


In [ ]:
# ---- the PRODUCTION log -> canonical log parquet (only v2 has one; ex-ingest.py) ----------
# Needs config paths.log_source + columns score/decision filled. Skips loudly until then.
try:
    src_path = config.path("log_source", "v2", "real")
    log_raw = pd.read_pickle(src_path) if str(src_path).endswith(".pkl") else pd.read_parquet(src_path)
    log = schema.to_canonical(log_raw, "v2")
    schema.require(log, "v2")               # names the config entry to fix if anything is missing
    dst = config.path("log", "v2", "real")
    dst.parent.mkdir(parents=True, exist_ok=True)
    log.to_parquet(dst, index=False)
    print("log ->", dst, len(log), "rows · scrap rate", float(log[schema.DECISION].mean()).__round__(4))
except (ValueError, FileNotFoundError, KeyError) as exc:
    print("log export SKIPPED —", exc)
    print("fill config.VERSIONS['v2']['paths']['log_source'] and columns score/decision, then re-run this cell.")


Notes:
- **Leave `paths.processed_inputs` / `paths.raw_dataset` undeclared in config** — analysis reads
  the parquet this notebook wrote at the fallback paths; that is the design.
- The recomputed scores are the pickled model on its own training matrix — in-sample for the
  train split. The **production log** is the only source of real decisions; the log cell above,
  not the scores cell, is what carries them.
